In [1]:
import pandas as pd

default_df = pd.read_csv(r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\default\1781332375_gasifier 30lpm 0.csv")


In [10]:
import os
import glob
import pandas as pd

def inspect_default_folder(file_path):
    """
    Dynamically locates the table header in Agilent 34970A files, loads the CSV,
    and outputs shape, column structure, and baseline statistics.
    """
    header_row_index = 0
    
    # Locate the telemetry table header row
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            if 'Scan Num' in line or '101 (' in line or 'Scan Swee' in line:
                header_row_index = idx
                break
                
    # Load dataset from the detected header row
    df = pd.read_csv(file_path, skiprows=header_row_index)
    
    # Clean column whitespace and drop completely empty rows or columns
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    return df

# ==========================================
# EXECUTION ON DEFAULT FOLDER
# ==========================================
default_path = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\default"
csv_files = glob.glob(os.path.join(default_path, "*.csv"))

print(f"Found {len(csv_files)} files in default folder. Running inspection...\n")

for i, file_path in enumerate(csv_files, 1):
    file_name = os.path.basename(file_path)
    
    try:
        df = inspect_default_folder(file_path)
        
        print(f"File {i}: {file_name}")
        print(f"   Shape: {df.shape[0]} rows by {df.shape[1]} columns")
        print(f"   Columns: {list(df.columns)}")
        print(f"   Missing Values: {df.isnull().sum().sum()} total nulls")
        
        # Select numeric columns for basic baseline stats
        numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
        if len(numeric_cols) > 0:
            print("\n   Baseline Data Summary (First 4 Numeric Columns):")
            print(df[numeric_cols[:4]].describe().loc[['mean', 'min', 'max', 'std']])
        
        print("\n" + "="*65 + "\n")
        
    except Exception as e:
        print(f"Could not read {file_name}: {e}\n")

Found 1 files in default folder. Running inspection...

File 1: 1781332375_gasifier 30lpm 0.csv
   Shape: 456 rows by 7 columns
   Columns: ['Scan Sweep Time (Sec)', 'Scan Number', '101 (°C)', '102 (°C)', '103 (°C)', '104 (°C)', '105 (°C)']
   Missing Values: 0 total nulls

   Baseline Data Summary (First 4 Numeric Columns):
      Scan Number     101 (°C)    102 (°C)    103 (°C)
mean    228.50000   674.904642  649.647536  694.076475
min       1.00000    29.560183   51.896423  595.116976
max     456.00000  1084.429830  902.305949  888.697890
std     131.78012   397.122003  211.825530   54.190740




In [9]:
import os
import glob
import pandas as pd

def load_34970a_sensor_data(file_path):
    """
    Dynamically locates the start of the telemetry table in an Agilent/Keysight 34970A 
    data log file and imports it into a clean Pandas DataFrame.
    """
    header_row_index = None
    
    # 1. Scan the file line-by-line to find where the actual data table begins
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            # We look for signature strings present in the telemetry header row
            if 'Scan Num' in line or '101 (' in line or 'Scan Swee' in line:
                header_row_index = idx
                break
                
    if header_row_index is None:
        raise ValueError(f"Could not locate telemetry data header in: {os.path.basename(file_path)}")
        
    # 2. Load the CSV into Pandas, skipping all metadata rows above the detected header
    df = pd.read_csv(file_path, skiprows=header_row_index)
    
    # 3. Clean up column names (strip unexpected spaces or formatting anomalies)
    df.columns = df.columns.str.strip()
    
    # 4. Drop any completely empty rows or columns (often caused by trailing commas in logger exports)
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    # 5. Standardize the Timestamp / Scan Sweep column name for consistency
    time_col_candidates = [col for col in df.columns if 'swee' in col.lower() or 'time' in col.lower()]
    if time_col_candidates:
        df.rename(columns={time_col_candidates[0]: 'Timestamp'}, inplace=True)
        
    return df

# ==========================================
# BATCH EXECUTION ON YOUR 'DEFAULT' FOLDER
# ==========================================
default_path = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\default"
csv_files = glob.glob(os.path.join(default_path, "*.csv"))

print(f"Successfully located {len(csv_files)} files. Beginning clean data extraction...\n")

for i, file_path in enumerate(csv_files, 1):
    file_name = os.path.basename(file_path)
    try:
        # Use our custom hardware loader
        df_clean = load_34970a_sensor_data(file_path)
        
        print(f"✅ File {i}: {file_name}")
        print(f"   • Clean Shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
        print(f"   • Extracted Headers: {list(df_clean.columns)}")
        print(f"   • First Row Sample Summary:\n{df_clean.iloc[0:2, :4]}\n")
        print("-" * 60)
        
    except Exception as e:
        print(f" Error loading {file_name}: {str(e)}\n")

Successfully located 1 files. Beginning clean data extraction...

✅ File 1: 1781332375_gasifier 30lpm 0.csv
   • Clean Shape: 456 rows × 7 columns
   • Extracted Headers: ['Timestamp', 'Scan Number', '101 (°C)', '102 (°C)', '103 (°C)', '104 (°C)', '105 (°C)']
   • First Row Sample Summary:
                 Timestamp  Scan Number   101 (°C)   102 (°C)
0  2024-05-14 12:18:23.620            1  29.936798  51.896423
1  2024-05-14 12:18:28.620            2  30.002421  52.132352

------------------------------------------------------------
